# SCORING NOTEBOOK 
- requires previous low_scorer and high_scorer runs

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import json
tqdm.pandas()

from xgboost import XGBRegressor

# Reproducibility
RANDOM_STATE = 42

# Target & features setup
TARGET_COL = "holistic_essay_score"
TEXT_COL = "text"
CATEGORICAL_COLS = ['gender', 'grade_level', 'race_ethnicity', 'economically_disadvantaged']

HIGH_META_PATH = "../outputs/model/run_01/high/feature_meta.json"
LOW_META_PATH  = "../outputs/model/run_01/low/feature_meta.json"

REWRITES_DIR = "../outputs/rewrites/sat/"
REWRITES_PREFIX = "rewrite_{}.csv"

EMB_DIR = "../outputs/embeddings/sat/"
EMB_PREFIX = "embeddings_rewrite_{}.npy"

SAVE_DIR = "../outputs/results/sat/"
SAVE_NAME = "data_sat_scored.csv"

MODEL_DIR = "../outputs/model/run_01/"

META_KEY_PATH = MODEL_DIR + "/high/feature_meta.json"

# Rewrites
REWRITE = "paraphrase"  # "paraphrase" or "random"

In [5]:
original = pd.read_csv(REWRITES_DIR + "original.csv")
original = original.drop(columns=original.filter(like="xgb_").columns)
original_emb = np.load(EMB_DIR + "embeddings_original.npy")

DATASET_REW = {"original": original}
DATASET_EMB = {"original": original_emb}

for i in range (1,7):
    DATASET_REW[i] = pd.read_csv(REWRITES_DIR+REWRITES_PREFIX.format(i))
    DATASET_EMB[i] = np.load(EMB_DIR+EMB_PREFIX.format(i))

/var/folders/lm/psfj50g95wgczw9z2l6zf32m0000gn/T/ipykernel_24865/4089930554.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  DATASET_REW[i] = pd.read_csv(REWRITES_DIR+REWRITES_PREFIX.format(i))
/var/folders/lm/psfj50g95wgczw9z2l6zf32m0000gn/T/ipykernel_24865/4089930554.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  DATASET_REW[i] = pd.read_csv(REWRITES_DIR+REWRITES_PREFIX.format(i))
/var/folders/lm/psfj50g95wgczw9z2l6zf32m0000gn/T/ipykernel_24865/4089930554.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  DATASET_REW[i] = pd.read_csv(REWRITES_DIR+REWRITES_PREFIX.format(i))
/var/folders/lm/psfj50g95wgczw9z2l6zf32m0000gn/T/ipykernel_24865/4089930554.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  DATASET_REW[i] = pd.read_csv(REWRITES_DIR+REWRITES_PREFIX

In [7]:
with open(META_KEY_PATH, "r") as f:
    feature_meta = json.load(f)

# Column order for the *tabular* part of each group
feature_cols_by_group = {
    "style":  feature_meta["style"],    # full style block (OHE + TAAL/TAACO/TAASSC)
    "emb":    feature_meta["emb_ohe"],  # OHE-only block used in embedding model
    "taaled": feature_meta["taaled"],
    "taaco":  feature_meta["taaco"],
    "taassc": feature_meta["taassc"],
    "full":   feature_meta["style"],    # full = style cols + embeddings
}

emb_dim = feature_meta["emb_dim"]

In [8]:
import uuid

# 1) sanity check: all DFs must have same length as `original`
n = DATASET_REW['original'].shape[0]

# 2) generate one UUID per paper
paper_ids = [str(uuid.uuid4()) for _ in range(n)]

# 3) assign paper_id to every DF
for df_ in list(DATASET_REW.values()):
    df_.loc[:, "paper_id"] = paper_ids

# 4) assign cv_fold from original to all rewritten datasets
# All dataframes have same row order, so we can copy cv_fold directly
for key, df_ in DATASET_REW.items():
    if key == "original":
        continue
    # Copy cv_fold from original (same row order due to same paper_ids)
    df_.loc[:, "cv_fold"] = original["cv_fold"].values

# 5) apply one-hot encoding to rewritten datasets
for key, df_ in DATASET_REW.items():
    if key == "original":
        continue 
    
    df_enc = pd.get_dummies(df_, columns=CATEGORICAL_COLS, drop_first=False)
    
    DATASET_REW[key] = df_enc

In [9]:
def load_xgb(path: Path) -> XGBRegressor:
    m = XGBRegressor()
    m.load_model(str(path))
    return m

# Map your logical group names to their folder names
SUBFOLDERS = {
    "full":   "x_full",
    "style":  "x_style",
    "emb":    "x_emb",
    "taassc": "x_taassc",
    "taaco":  "x_taaco",
    "taaled": "x_taaled",
}

def build_models(root=None, folds=5):
    root = Path(root)
    return {
        group: {
            side: {
                i: load_xgb(root / f"{side}" / sub / f"xgb_fold{i}.json")
                for i in range(1, folds + 1)
            }
            for side in ("high", "low")
        }
        for group, sub in SUBFOLDERS.items()
    }

models = build_models(MODEL_DIR)

In [10]:
name_to_k = {
    "original": 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5,
    6: 6,
}

In [11]:
from utils.compare_df import compare_columns

compare_columns(DATASET_REW['original'], DATASET_REW[1], 'og', 'rew')

Columns only in og (0): []
Columns only in rew (2): ['content_preserved', 'rewritten_text']
Common columns (376). Example: ['text', 'holistic_essay_score', 'prompt_name', 'gender_F', 'gender_M', 'grade_level_6.0', 'grade_level_8.0', 'grade_level_9.0', 'grade_level_10.0', 'grade_level_11.0']


In [12]:
groups = ["full", "style", "emb", "taaled", "taaco", "taassc"]

rows = []

for name, df_ in DATASET_REW.items():

    # --- Build feature DF x (keep df_ unchanged)
    if name != "original":
        x = df_.drop(
            columns=[
                "full_text", "text_tokens", "content_preserved",
                "rewritten_text", "text"
            ],
            errors="ignore"
        ).copy()
    else:
        x = df_.drop(
            columns=["full_text", "text_tokens", "text"],
            errors="ignore"
        ).copy()

    # --- Keep references for output
    true_score = x["holistic_essay_score"].copy()
    paper_ids  = x["paper_id"].copy()
    low_ses    = x["economically_disadvantaged_1"].copy()
    race       = x["race_ethnicity_White"].copy()
    gender     = x["gender_M"].copy()
    prompt     = x["prompt_name"].copy()

    grade_cols = [c for c in x.columns if c.startswith("grade_level_")]
    grade      = (
        x[grade_cols]
        .idxmax(axis=1)
        .str.replace("grade_level_", "")
        .astype(float)
    )

    cv_fold = x["cv_fold"].astype(int).copy()

    # --- Remove label/meta cols from features
    x = x.drop(
        columns=["paper_id", "cv_fold", "holistic_essay_score", "prompt_name"],
        errors="ignore"
    )

    # --- Optional sanity check: all required cols exist
    x_cols = set(x.columns)
    for g in ["style", "emb", "taaled", "taaco", "taassc", "full"]:
        missing = set(feature_cols_by_group[g]) - x_cols
        if missing:
            raise ValueError(f"Missing columns for group '{g}' in dataset '{name}': {missing}")

    # --- Build feature matrices for each scorer using saved order
    X_emb = DATASET_EMB[name]  # embedding matrix aligned row-wise to df_

    X_by_group = {
        "style":  x[feature_cols_by_group["style"]].to_numpy(),
        "full":   np.hstack([
                     x[feature_cols_by_group["full"]].to_numpy(),
                     X_emb
                 ]),
        "emb":    np.hstack([
                     x[feature_cols_by_group["emb"]].to_numpy(),
                     X_emb
                 ]),
        "taaled": x[feature_cols_by_group["taaled"]].to_numpy(),
        "taaco":  x[feature_cols_by_group["taaco"]].to_numpy(),
        "taassc": x[feature_cols_by_group["taassc"]].to_numpy(),
    }

    # --- Prepare output arrays for all groups/sides
    preds = {
        g: {
            "high": np.empty(len(x), float),
            "low":  np.empty(len(x), float),
        }
        for g in groups
    }

    # --- Predict per fold for each group
    for f in (1, 2, 3, 4, 5):
        mask = (cv_fold == f)
        if not np.any(mask):
            continue
        for g in groups:
            Xf = X_by_group[g][mask]
            preds[g]["high"][mask] = models[g]["high"][f].predict(Xf)
            preds[g]["low"][mask]  = models[g]["low"][f].predict(Xf)

    # --- Assemble row block with 14 score columns
    out = {
        "essay_id":    paper_ids.values,
        "k":           np.full(len(paper_ids), name_to_k[name], dtype=int),
        "true_score":  true_score.values,
        "low_SES":     low_ses.values,
        "race_white":  race.values,
        "gender_male": gender.values,
        "prompt_name": prompt.values,
        "grade_level": grade.values,
        "cv_fold":     cv_fold.values,
    }
    for g in groups:
        out[f"score_high_{g}"] = preds[g]["high"]
        out[f"score_low_{g}"]  = preds[g]["low"]

    rows.append(pd.DataFrame(out))

results_df = pd.concat(rows, ignore_index=True)


In [16]:
results_df.to_csv(f"{SAVE_DIR}{SAVE_NAME}")

In [ ]:
results_df.shape

In [17]:
results_df.head()

,essay_id,k,true_score,low_SES,race_white,gender_male,prompt_name,grade_level,cv_fold,score_high_full,...,score_high_style,score_low_style,score_high_emb,score_low_emb,score_high_taaled,score_low_taaled,score_high_taaco,score_low_taaco,score_high_taassc,score_low_taassc
0,0d717d5d-2734-4d00-98ab-8527a4d664f7,0,3,False,True,True,Summer projects,11.0,3,3.998649,...,3.925160,3.859234,3.848287,3.718099,3.815336,3.503736,3.569814,3.745397,3.833944,3.900998
1,48c17d01-a804-4ead-933c-b8cb18777a21,0,4,False,False,True,Summer projects,11.0,1,3.739981,...,3.676664,3.401469,3.379174,3.548945,3.863222,3.703911,4.162063,3.825643,3.840232,3.497867
2,a20e23d7-2ad8-4656-a055-1fe5bbd35ecb,0,4,False,True,True,Summer projects,11.0,4,4.987003,...,5.148112,5.121298,3.998368,3.851926,4.701632,4.711694,5.371764,4.993152,4.924241,5.019535
3,f60f0f95-0182-4d83-a385-0c03cdbae761,0,4,False,False,False,Summer projects,11.0,2,4.593962,...,4.543294,4.464949,4.619034,4.593761,4.583718,4.099163,4.635283,4.419876,4.542449,4.473258
4,03ded759-6591-4d30-aa47-b45a5cc48815,0,3,True,False,True,Summer projects,11.0,1,4.088375,...,3.403682,4.266418,4.545832,4.627963,4.023148,3.787030,3.568487,3.738469,3.926355,4.186929


In [15]:
results_df['true_score'].mean()

np.float64(3.3246246246246245)

In [ ]:
results_df = results_df[~results_df['grade_level'].isin([12.0, 9.0])]

In [27]:
results_df.shape

(137095, 21)

In [28]:
results_df.to_csv(f"{SAVE_DIR}data_sat_scored_cleaned.csv")